# Bobathon ESILV — Parkinson's True Motor Score Prediction
### ESILV Major — Machine Learning Course (2026-2027)
### Instructor: Prof. Hugo ALATRISTA SALAS

---

### Team Composition
- **Team Name (Kaggle):** `parKINGson`
- **Team Members:**
  1. **Damien CATAU**
  2. **Malo GERVAISE**
  3. **Esteban GAUBERT**
  4. **Maximilien STUDER**

---

> ℹ️ **Note on Detailed Experiment Documentation in `journal/`:**
> This notebook synthesizes our entire iterative machine learning pipeline, exploratory data analysis, experimental evaluations, and answers to all workbook questions.
> **For complete mathematical derivations, architecture blueprints, hyperparameter grids, and ablation logs of each individual iteration, please refer directly to the markdown files in the `journal/` directory** (`journal/01_dummy.md` through `journal/13_super_moe_final_sota.md` and `journal/JOURNAL.md`).

---

### Challenge Context & Clinical Objectives
Parkinson's Disease (PD) is a progressive neurodegenerative disorder characterized by motor symptoms such as tremors, muscle rigidity, bradykinesia (slowness of movement), and postural instability. Clinically, motor impairment is quantified using the **MDS-UPDRS Part III (Motor Examination)** score, ranging from 0 (no motor disability) to 132 (extreme disability).

Patients undergoing dopaminergic therapy (e.g., levodopa) oscillate between two clinical states:
1. **ON State**: Symptoms are suppressed while medication is therapeutically active in plasma.
2. **OFF State**: Symptoms reappear as medication concentration drops below the therapeutic threshold.

While the **OFF score** is the true indicator of underlying neurodegeneration, clinical OFF evaluations suffer from severe observational noise:
- **Intake timing delays**: Variable oral absorption ($k_a$) and clearance ($k_e$) mean tests rarely capture the true peak OFF state.
- **Rater subjectivity**: 50 clinical raters introduce systematic scoring leniency or strictness.
- **Informative missingness**: 42.4% of visits have no recorded OFF score because medication withdrawal is physically exhausting and frequently skipped for frail patients.

**Target:** Predict the debiased, unobserved **true OFF score** (`target`) for every patient visit across an unseen patient holdout set.


## 1. Environment Setup & Data Ingestion

We begin by initializing our data environment and loading the competition tables from `data/`:
- `X_train.csv` & `y_train.csv`: 44,590 clinical visits across 5,576 unique patients.
- `X_test.csv`: 11,013 clinical visits across 1,395 unseen patients (strictly zero patient overlap).
- `sample_submission.csv`: Competition submission template (`Index,target`).


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure local modular package 'src/' is available in Python path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Set visual style for publication-ready figures
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 11})

# Load raw competition datasets
X_train = pd.read_csv("data/X_train.csv")
y_train = pd.read_csv("data/y_train.csv")
X_test = pd.read_csv("data/X_test.csv")

# Merge features and ground-truth target for training visits
visits = X_train.merge(y_train, on="Index")

print(f"Training dataset: {len(visits):,} visits across {visits['patient_id'].nunique():,} unique patients")
print(f"Test dataset:     {len(X_test):,} visits across {X_test['patient_id'].nunique():,} unique patients")
print(f"Features:         {visits.shape[1] - 2} columns (excluding Index and target)")


## 2. Exploratory Data Analysis & Day 1 Workbook Answers

> 📖 *For deeper interactive profiling, see `data/eda.md`, `data/eda_train.html`, and `journal/01_dummy.md`.*

### Detailed Answers to Day 1 Workbook Questions

#### Question 1: Patient visit frequency and cohort distribution
- **Training visits:** 44,590 visits across 5,576 distinct patients.
- **Test visits:** 11,013 visits across 1,395 distinct patients.
- **Visits per patient:** Median is 7 visits per patient (IQR: 5–9, minimum 1, maximum 12).
- **Core ML Implication:** Patients appear repeatedly across visits. Because the test set contains entirely new patients, **random row splitting causes massive data leakage**. All validation must be grouped by `patient_id` using `GroupKFold`.

#### Question 2: Missingness patterns and clinical exam skipping
- **Top 3 most missing columns:**
  1. `time_since_intake_off`: **78.8% missing** (delay since last dose before OFF assessment).
  2. `time_since_intake_on`: **46.4% missing** (delay since last dose before ON assessment).
  3. `off`: **42.4% missing** (clinic-measured OFF motor score).
  4. `ledd`: **36.6% missing** (levodopa equivalent daily dose).
  5. `gene`: **32.4% missing** (genetic mutation marker).
- **Visits with no `off` score:** 42.4% of visits.
- **Visits with no `on` score:** 29.6% of visits.
- **Visits with both `on` and `off` missing:** Observed in ~14.2% of visits.
- **Clinical Insight:** Missingness in `off` is **informative signal**, not missing completely at random (MCAR). Frail patients with advanced neurodegeneration cannot endure levodopa withdrawal. Tree models that branch natively on `NaN` leverage this signal.

#### Question 3: Target distribution vs. Theoretical scale
- **Observed Range:** $[0.0, 109.5]$ (theoretical clinical maximum of MDS-UPDRS Part III is 132).
- **Distribution Shape:** Bell-shaped with moderate right-tail dispersion, Mean = 37.5, Median = 37.3, Std = 16.5.
- **Error Floor:** A standard deviation of $\approx 16.5$ establishes the theoretical RMSE of a naive dummy mean predictor.

#### Question 4: Feature associations with Target
- **`off` (measured OFF):** Pearson correlation $r = 0.871$ ($V = 0.406$). It is the single strongest linear predictor, but absent in 42.4% of visits.
- **`on` (measured ON):** Second strongest linear predictor ($r = 0.669$).
- **`age` & `ledd`:** Moderate correlation ($r = 0.310$ and $r = 0.298$).
- **Intake delays:** Near-zero linear Pearson $r (< 0.01)$, but non-linear association (Cramér's $V = 0.177$), pointing to non-linear pharmacokinetic curve dynamics.

#### Question 5: Medication dose (`ledd`) across cohorts
- In Cohort A, `ledd` is missing in only ~12% of visits, whereas in Cohort B it is missing in >60% of visits. This indicates disparate recording protocols between clinical centers.

#### Question 6: Features excluded from modeling
- `Index`: Database primary key.
- `patient_id`: Group identifier. Including it causes high-cardinality memorization without generalization to unseen test patients.


In [ ]:
# Visualizing Target Distribution and Missingness Breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Target distribution
sns.histplot(visits["target"], kde=True, color="#2b5c8f", bins=40, ax=axes[0])
axes[0].axvline(visits["target"].mean(), color="red", linestyle="--", label=f"Mean: {visits['target'].mean():.1f}")
axes[0].axvline(132, color="black", linestyle=":", label="Theoretical Max (132)")
axes[0].set_title("True OFF MDS-UPDRS Distribution (target)")
axes[0].set_xlabel("Motor Score")
axes[0].set_ylabel("Visit Frequency")
axes[0].legend()

# Plot 2: Missingness by column
missing = (visits.isnull().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]
sns.barplot(x=missing.values, y=missing.index, palette="mako", ax=axes[1])
axes[1].set_title("Feature Missingness Rates (%)")
axes[1].set_xlabel("Missing Percentage (%)")

plt.tight_layout()
plt.show()


In [ ]:
# Visualizing Measured OFF vs True Target (Clinical Debiasing Goal)
plt.figure(figsize=(8, 6))
sample = visits.dropna(subset=["off"]).sample(3000, random_state=42)
sns.scatterplot(data=sample, x="off", y="target", hue="time_since_intake_off", palette="viridis", alpha=0.6)
plt.plot([0, 110], [0, 110], color="red", linestyle="--", label="Ideal Identity (off == target)")
plt.title("Measured Clinic OFF vs True Debiaised OFF Target (r = 0.871)")
plt.xlabel("Measured Clinic OFF Score")
plt.ylabel("True Unbiased Target Score")
plt.legend()
plt.show()


## 3. Baseline: Dummy Mean Model (The Reference Floor)

> 📖 *For complete design notes and Skore Hub tracking details, see `journal/01_dummy.md`.*

### Purpose & Clinical Logic
A `DummyRegressor(strategy="mean")` ignores all visit covariates and predicts the constant empirical mean of the training target:
$$\hat{y}_i = \bar{y} = \frac{1}{N}\sum_{i=1}^N y_i \approx 37.5$$

- **Error Floor:** Yields an RMSE $\approx 16.48$.
- **Sanity check:** Any trained machine learning model must substantially beat 16.48 to prove that its features extract meaningful clinical signal.


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error

FEATURE_COLS = [
    "sexM", "age_at_diagnosis", "age", "ledd",
    "time_since_intake_on", "time_since_intake_off",
    "on", "off"
]

X_base = visits[FEATURE_COLS]
y_base = visits["target"]

# Evaluate dummy baseline
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_base, y_base)
dummy_preds = dummy.predict(X_base)
dummy_rmse = root_mean_squared_error(y_base, dummy_preds)

print(f"Dummy Mean Baseline RMSE: {dummy_rmse:.4f}")
print(f"Target Standard Deviation: {y_base.std():.4f}")
print("-> Matches expected mathematical identity: RMSE(mean) == std(y)")


## 4. Linear Model: Ridge Regression & Alpha Regularization

> 📖 *For full tuning logs and coefficient comparisons, see `journal/02_ridge.md`.*

### Answers to Day 2 Step 1 & 2 Workbook Questions

#### 1. Why Imputation is Mandatory for Ridge
Linear models compute weighted sums: $\hat{y} = w^T x + b$. A single `NaN` produces an undefined result. We must apply `SimpleImputer(strategy="median")` to fill missing numeric features.

#### 2. By what percentage does Ridge lower the RMSE compared with Dummy?
Ridge lowers the RMSE from **16.48** down to **10.60**, representing a **35.7% relative reduction in prediction error**. This is a massive win, demonstrating that raw clinical measures (`off`, `on`, `age`, `ledd`) carry strong additive predictive power.

#### 3. Alpha Regularization Sensitivity
We sweep regularisation strength $\alpha \in [0.1, 1.0, 10.0, 100.0, 1000.0]$.
- **Finding:** With 44,590 training rows and only 8 features, the sample-to-feature ratio exceeds $5,500:1$. The system is heavily overdetermined; thus, varying $\alpha$ between 0.1 and 100.0 changes the 5-fold RMSE by less than $0.005$.
- Optimal performance occurs at $\alpha = 100.0$.

#### 4. Interpretation of Coefficients & Unscaled Features Caveat
Because features were not standardized, raw coefficients cannot be interpreted directly as relative importance:
- An increase of 1 year in `age` contributes differently from 1 unit of `ledd` (mg) or `off` (scale 0-100).
- However, signs confirm clinical reality: `off` weight is $+0.72$ (positive anchor) and `on` weight is $+0.18$.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold

groups = visits["patient_id"]
gkf = GroupKFold(n_splits=5)

alpha_values = [0.1, 1.0, 10.0, 100.0, 1000.0]
ridge_results = {}

print("--- Ridge Alpha Sweep with 5-Fold GroupKFold ---")
for alpha in alpha_values:
    pipe = make_pipeline(SimpleImputer(strategy="median"), Ridge(alpha=alpha))
    fold_rmses = []
    for train_idx, val_idx in gkf.split(X_base, y_base, groups=groups):
        pipe.fit(X_base.iloc[train_idx], y_base.iloc[train_idx])
        preds = pipe.predict(X_base.iloc[val_idx])
        fold_rmses.append(root_mean_squared_error(y_base.iloc[val_idx], preds))
    mean_score = np.mean(fold_rmses)
    ridge_results[alpha] = mean_score
    print(f"Alpha {alpha:>8.1f} | 5-Fold Mean RMSE: {mean_score:.4f}")

best_alpha = min(ridge_results, key=ridge_results.get)
print(f"\nWinning Ridge Alpha: {best_alpha} (RMSE = {ridge_results[best_alpha]:.4f})")


## 5. Honest Evaluation: Patient-Grouped CV (`GroupKFold`)

> 📖 *For comparative analysis on leakage, see `journal/03_hgbr.md`.*

### Answers to Day 2 Step 3 Workbook Questions

#### 1. Compare grouped-CV RMSE with random-split RMSE
- **Random Row Holdout (leakage):** Reported Ridge RMSE $\approx \mathbf{10.12}$.
- **Patient-Grouped CV (`GroupKFold`):** Honest Ridge RMSE $\approx \mathbf{10.60}$.
- **Why the difference?**
  A random row split distributes visits from the same patient across both training and validation folds. The model learns to memorize patient-specific offsets rather than generalizing disease dynamics.

#### 2. Which estimate is closer to Kaggle public score?
The competition test set (`X_test.csv`) strictly holds out entire unseen patients. Consequently, **random-split scores are over-optimistic illusions**. The `GroupKFold(n_splits=5, groups=patient_id)` score mirrors the true Kaggle holdout distribution and must be used for all subsequent decisions.


## 6. Tree-based Modeling: HistGradientBoosting (Native NaNs)

> 📖 *For complete fold-by-fold metrics and tree parameters, see `journal/03_hgbr.md`.*

### Answers to Day 2 Step 4 Workbook Questions

#### 1. Why Native NaN Routing Beats Imputation
Median imputation forces missing `off` scores to $\approx 37.0$, falsely declaring that patients who skipped the test are average. In reality, skipping an OFF exam is strongly correlated with advanced disease severity.
`HistGradientBoostingRegressor` learns optimal branching directions for `NaN` values directly at each tree split:
- **Result:** HGBR drops the cross-validation RMSE from **10.60 (Ridge)** down to **~7.75**.
- **Fold-by-fold check:** HGBR beats Ridge decisively on **every single fold** ($p < 0.001$).

#### 2. Permutation Importance
Feature importance ranking reveals:
1. `off`: 74.2% of total permutation importance.
2. `on`: 16.5% of importance.
3. `age` & `disease_duration`: 6.1% of importance.
4. `ledd` & intake delays: 3.2% of importance.


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgbr = HistGradientBoostingRegressor(random_state=42)
hgbr_fold_rmses = []

for train_idx, val_idx in gkf.split(X_base, y_base, groups=groups):
    hgbr.fit(X_base.iloc[train_idx], y_base.iloc[train_idx])
    preds = hgbr.predict(X_base.iloc[val_idx])
    hgbr_fold_rmses.append(root_mean_squared_error(y_base.iloc[val_idx], preds))

hgbr_rmse = np.mean(hgbr_fold_rmses)
print(f"HGBR (Native NaNs) 5-Fold GroupKFold RMSE: {hgbr_rmse:.4f}")
print(f"Net gain over Ridge: {ridge_results[best_alpha] - hgbr_rmse:.4f} RMSE points")


## 7. Automated Table Vectorization with `skrub`

> 📖 *For encoding diagnostics and cardinality analysis, see `journal/04_skrub_tabular.md`.*

### Answers to Day 2 Step 5 Workbook Questions

#### 1. Handling Mixed Types without Pipeline Rot
Prior experiments only processed numeric columns, discarding `gene`, `cohort`, and `rater_id`.
`skrub.TableVectorizer` automatically detects column cardinality:
- **Low Cardinality (`cohort`, `sexM`):** One-hot encoded into binary indicator columns.
- **High Cardinality (`gene`, `rater_id`):** Vectorized via compact `StringEncoder` representations.
- **Continuous Numerics:** Passed through unchanged, preserving `NaN` values for HGBR.

#### 2. Quantitative Gain
Adding categorical columns reduces RMSE from **7.75** to **7.53**.


In [ ]:
from skrub import TableVectorizer

X_all = visits.drop(columns=["Index", "patient_id", "target"])

skrub_pipeline = make_pipeline(
    TableVectorizer(),
    HistGradientBoostingRegressor(random_state=42)
)

skrub_scores = []
for train_idx, val_idx in gkf.split(X_all, y_base, groups=groups):
    skrub_pipeline.fit(X_all.iloc[train_idx], y_base.iloc[train_idx])
    preds = skrub_pipeline.predict(X_all.iloc[val_idx])
    skrub_scores.append(root_mean_squared_error(y_base.iloc[val_idx], preds))

skrub_rmse = np.mean(skrub_scores)
print(f"skrub TableVectorizer + HGBR 5-Fold RMSE: {skrub_rmse:.4f}")


## 8. Declarative Machine Learning: `skrub DataOps`

> 📖 *For graph execution tracing, see `journal/05_dataops.md`.*

### Answers to Day 2 Step 6 Workbook Questions

#### Why DataOps?
Separating data frames, cross-validation splitters, and models into independent variables creates risk of leakage and alignment bugs.
**DataOps** attaches the cross-validation strategy (`GroupKFold`) and grouping variable (`patient_id`) directly to the computational graph node (`mark_as_X`).
- The split cannot drift or detach from the features.
- At inference time, `SkrubLearner` automatically executes the frozen transformation graph on `X_test`.


## 9. Clinical Feature Engineering (Domain Signal)

> 📖 *For mathematical derivations and feature ablations, see `journal/06_features.md` and `journal/07_tuning.md`.*

### Engineered Clinical Features (`src/parkinson/features.py`)
1. **Disease Duration:** $\text{disease\_duration} = \text{age} - \text{age\_at\_diagnosis}$.
2. **Bateman Pharmacokinetics (Levodopa Plasma Kinetics):**
   $$C(t) = \frac{k_a}{k_a - k_e} \left( e^{-k_e t} - e^{-k_a t} \right)$$
   Where absorption constant $k_a = 1.5\text{ h}^{-1}$ and elimination constant $k_e = 0.4\text{ h}^{-1}$ over `time_since_intake_on/off`.
3. **Motor Responsiveness Delta:** $\text{on\_off\_gap} = \text{off} - \text{on}$, capturing patient responsiveness to dopaminergic therapy.
4. **Visit Chronology & Frequency:** Total visits and visit index per patient.
5. **Explicit Missingness Flags:** `off_is_missing`, `on_is_missing`, `both_missing`.

### Result
Clinical feature engineering causes a **major performance leap**, driving RMSE from **7.53** down to **5.68**!


In [ ]:
from parkinson.features import build_features

# Apply stateless feature engineering
visits_fe = build_features(visits)
X_fe = visits_fe.drop(columns=["Index", "patient_id", "target"])

fe_pipeline = make_pipeline(TableVectorizer(), HistGradientBoostingRegressor(random_state=42))

fe_scores = []
for train_idx, val_idx in gkf.split(X_fe, y_base, groups=groups):
    fe_pipeline.fit(X_fe.iloc[train_idx], y_base.iloc[train_idx])
    preds = fe_pipeline.predict(X_fe.iloc[val_idx])
    fe_scores.append(root_mean_squared_error(y_base.iloc[val_idx], preds))

fe_rmse = np.mean(fe_scores)
print(f"Clinical Features + HGBR 5-Fold RMSE: {fe_rmse:.4f}")
print(f"Gain from Domain Features: {skrub_rmse - fe_rmse:.4f} RMSE points")


## 10. Advanced SOTA Architectures: Breaking Below 3.10 RMSE

> 📖 *For complete technical design notes and code of advanced architectures, see:*
> - `journal/08_clinical_gbdt.md`: Dual-Branch LightGBM + Rater Calibration
> - `journal/09_sequence_transformer.md`: Bi-Directional SeqNet with Stochastic Masking
> - `journal/10_retrospective_smoothing.md`: Calibrated Retrospective Smoothing Blend
> - `journal/11_monotonic_bernstein_trajectory.md`: Monotone Trajectory Projection
> - `journal/12_multi_paradigm_bernstein_sota.md`: Time-Decay SeqNet + Anchored GBDT + Bernstein
> - `journal/13_super_moe_final_sota.md`: Super-Mixture-of-Experts (SOTA)

### Key Architectural Breakthroughs:

#### 1. Dual-Branch Routing (`DualBranchGBDT` in `src/parkinson/dual_branch.py`)
Visits with observed OFF vs. missing OFF represent completely distinct clinical regimes. Dual-branch architecture routes samples into specialized estimators:
- **Branch A (OFF observed):** Predicts residual corrections on top of measured OFF.
- **Branch B (OFF missing):** Leverages longitudinal trajectories and ON dynamics.
- **Rater Calibration (`RaterCalibrator`):** Bayesian-shrinkage target encoding eliminates grading bias across 50 raters.

#### 2. Bi-Directional Sequence Transformer (`SeqNet` in `src/parkinson/seqnet.py`)
Because test visits for each patient are available simultaneously (retrospective setting), `SeqNet` processes each patient's entire visit history through multi-head self-attention:
- Sinusoidal continuous-time age embeddings.
- Stochastic OFF masking ($p=0.35$) forces the model to reconstruct missing OFF scores across the timeline.
- **Result:** Plunges pooled RMSE from **5.57** to **3.308**!

#### 3. Monotonic Bernstein Trajectory Projector (`src/parkinson/trajectory.py`)
Neurodegeneration in Parkinson's disease is biologically non-reversible: true baseline motor scores cannot spontaneously decrease over years.
We fit a constrained degree-3 Bernstein polynomial per patient:
$$\hat{y}(s) = \sum_{k=0}^3 \theta_k B_{k, 3}(s), \quad \text{where } \theta = M \alpha \text{ with } \alpha_{1..3} \ge 0$$
- Mathematically guarantees $\frac{d\hat{y}}{d(\text{age})} \ge 0$.
- Eliminates 100% of trajectory monotonicity violations.
- Coupled with continuous-time decay attention (`12`), reaches our current top validated score of **3.0135 OOF RMSE**!


## 11. Complete Experimental Leaderboard & Benchmark Progression

The progression across our 13 experimental milestones:

| # | Experiment Script | Architecture & Key Methodology | 5-Fold GroupKFold RMSE | Key Takeaway | Companion Design Note |
|:---:|:---|:---|:---:|:---|:---:|
| 01 | `01_dummy.py` | Mean Baseline | 16.48 | Reference error floor (matching target standard deviation) | `journal/01_dummy.md` |
| 02 | `02_ridge.py` | Ridge + Median Imputation ($\alpha=100.0$) | 10.60 | Linear baseline; proves clinical columns carry signal | `journal/02_ridge.md` |
| 03 | `03_hgbr.py` | HistGradientBoosting (Native NaNs) | 7.75 | Missingness in OFF exam is informative signal | `journal/03_hgbr.md` |
| 04 | `04_skrub_tabular.py` | TableVectorizer + HGBR | 7.53 | Automated encoding of categoricals (`gene`, `cohort`) | `journal/04_skrub_tabular.md` |
| 05 | `05_dataops.py` | skrub DataOps declarative graph | 7.53 | Bakes patient groups and CV onto computation graph | `journal/05_dataops.md` |
| 06 | `06_features.py` | 9 Domain Clinical Features + HGBR | 5.68 | Levodopa dynamics, disease duration, ON/OFF delta | `journal/06_features.md` |
| 07 | `07_tuning.py` | Tuned HGBR + Two-Seed Ensemble | 5.57 | Regularization and variance reduction via ensembling | `journal/07_tuning.md` |
| 08 | `08_clinical_gbdt.py` | Dual-Branch LightGBM + Bateman PK + Rater Calibration | 6.04 | Specialized branches for OFF-observed vs. missing | `journal/08_clinical_gbdt.md` |
| 09 | `09_sequence_transformer.py` | Bi-Directional SeqNet (Self-Attention + Masking) | 3.308 | Timeline smoothing; reconstructs missing OFF | `journal/09_sequence_transformer.md` |
| 10 | `10_retrospective_smoothing.py` | Calibrated Dual-Regime Convex Blend (SeqNet + GBDT) | 3.303 | Blends deep sequence representations with sharp tree splits | `journal/10_retrospective_smoothing.md` |
| 11 | `11_monotonic_bernstein.py` | Monotonic Bernstein Trajectory Projector | 3.231 | Enforces physiologically non-decreasing motor progression | `journal/11_monotonic_bernstein_trajectory.md` |
| 12 | `12_multi_paradigm_sota.py` | TD-SeqNet + Anchored GBDT + 6-Regime SLSQP + Bernstein | **3.0135** | Winning architecture: continuous-time physics + Bernstein | `journal/12_multi_paradigm_bernstein_sota.md` |
| 13 | `13_super_moe_final_sota.py` | Super-MoE (SeqNet v1 + TD-SeqNet) + Bernstein | **~2.99** | Dual-Transformer mixture-of-experts | `journal/13_super_moe_final_sota.md` |


In [ ]:
# Visualizing Model Evolution: From Baseline to SOTA (RMSE Reduction)
experiments = [
    "01 Dummy", "02 Ridge", "03 HGBR", "04 skrub", "06 Features",
    "07 Tuned", "09 SeqNet", "10 RSR Blend", "11 Bernstein", "12 TD-SeqNet SOTA"
]
rmses = [16.48, 10.60, 7.75, 7.53, 5.68, 5.57, 3.308, 3.303, 3.231, 3.0135]

plt.figure(figsize=(12, 6))
bars = plt.bar(experiments, rmses, color=sns.color_palette("flare", len(experiments)))
plt.axhline(16.48, color="red", linestyle="--", label="Baseline Floor (16.48)")
plt.axhline(3.0135, color="green", linestyle=":", label="Current SOTA (3.0135)")

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.25, f"{yval:.2f}", ha='center', va='bottom', fontsize=10, weight='bold')

plt.title("Error Progression Across Experiments (5-Fold Patient Grouped RMSE)", fontsize=14, weight='bold')
plt.xlabel("Experiment Milestone")
plt.ylabel("Cross-Validation RMSE")
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 18.5)
plt.legend()
plt.tight_layout()
plt.show()


## 12. Final Submission Verification

We verify that the final submission file (`submissions/best_submission.csv`):
1. Matches the required shape: exactly 11,013 rows and 2 columns (`Index,target`).
2. Contains no `NaN` or infinite values.
3. Strictly adheres to valid clinical bounds for MDS-UPDRS Part III: $\hat{y} \in [0, 132]$.


In [ ]:
# Inspect and validate the final submission file
sub_path = Path("submissions/best_submission.csv")

if not sub_path.exists():
    print("Generating best_submission.csv from winning experiment...")
    import importlib
    best_module = importlib.import_module("experiments.00_best_model")
    best_module.run()

sub = pd.read_csv(sub_path)

print(f"File:        {sub_path.name}")
print(f"Shape:       {sub.shape} (Expected: (11013, 2))")
print(f"Nulls:       {sub.isnull().sum().to_dict()}")
print(f"Range:       [{sub['target'].min():.2f}, {sub['target'].max():.2f}] (Allowed bounds: [0, 132])")
assert len(sub) == 11013, f"Expected 11,013 rows, got {len(sub)}"
assert list(sub.columns) == ["Index", "target"], f"Columns mismatch: {sub.columns}"
assert not sub.isnull().any().any(), "Submission contains null values!"
assert sub["target"].between(0, 132).all(), "Predictions out of physical bounds!"
print("\n[SUCCESS] Final submission file is fully valid and ready for Kaggle / DVL upload!")

# Display preview of first 5 predictions
sub.head()
